Use the famous boston dataset and find out if there is multicollinearity or not. If there is multicollinearity try to remove it with help of steps taught in the session.

In [ ]:
# Link - https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv')

In [3]:
df.shape

(506, 14)

In [4]:
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [ ]:
import seaborn as sns

sns.heatmap(df.corr(method='pearson'),annot=True)

In [6]:
df.corr()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
crim,1.000000,-0.200469,0.406583,-0.055892,0.420972,-0.219247,0.352734,-0.379670,0.625505,0.582764,0.289946,-0.385064,0.455621,-0.388305
zn,-0.200469,1.000000,-0.533828,-0.042697,-0.516604,0.311991,-0.569537,0.664408,-0.311948,-0.314563,-0.391679,0.175520,-0.412995,0.360445
indus,0.406583,-0.533828,1.000000,0.062938,0.763651,-0.391676,0.644779,-0.708027,0.595129,0.720760,0.383248,-0.356977,0.603800,-0.483725
chas,-0.055892,-0.042697,0.062938,1.000000,0.091203,0.091251,0.086518,-0.099176,-0.007368,-0.035587,-0.121515,0.048788,-0.053929,0.175260
nox,0.420972,-0.516604,0.763651,0.091203,1.000000,-0.302188,0.731470,-0.769230,0.611441,0.668023,0.188933,-0.380051,0.590879,-0.427321
rm,-0.219247,0.311991,-0.391676,0.091251,-0.302188,1.000000,-0.240265,0.205246,-0.209847,-0.292048,-0.355501,0.128069,-0.613808,0.695360
age,0.352734,-0.569537,0.644779,0.086518,0.731470,-0.240265,1.000000,-0.747881,0.456022,0.506456,0.261515,-0.273534,0.602339,-0.376955
dis,-0.379670,0.664408,-0.708027,-0.099176,-0.769230,0.205246,-0.747881,1.000000,-0.494588,-0.534432,-0.232471,0.291512,-0.496996,0.249929
rad,0.625505,-0.311948,0.595129,-0.007368,0.611441,-0.209847,0.456022,-0.494588,1.000000,0.910228,0.464741,-0.444413,0.488676,-0.381626
tax,0.582764,-0.314563,0.720760,-0.035587,0.668023,-0.292048,0.506456,-0.534432,0.910228,1.000000,0.460853,-0.441808,0.543993,-0.468536


In [7]:
df.columns

Index(['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'b', 'lstat', 'medv'],
      dtype='object')

In [17]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


df1 = df.drop('medv',axis=1)
df1.columns

df1 = add_constant(df1)
# df1.columns
ouput_df = df['medv']
vifs = []
for i in range(df1.shape[1]):
	vif = variance_inflation_factor(df1.values,i)
	vifs.append(vif)

vif_df = pd.DataFrame({
	'feature' : df1.columns,
	'vifs' : vifs
})

vif_df

,feature,vifs
0,const,1.000000
1,crim,1.792192
2,zn,2.298758
3,indus,3.991596
4,chas,1.073995
5,nox,4.393720
6,rm,1.933744
7,age,3.100826
8,dis,3.955945
9,rad,7.484496


***tax and rad has multicollinearity above the kept threshold level of 5.0 therefore we need to remove multicollinearity from them***

In [19]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(df1.drop('const',axis=1))

np.linalg.cond(X_scaled)

np.float64(9.822002823124395)

## We can decrease multi collinearity by dropping the col or pca or regularization or by collecting more data

* The simple version is to drop tax first, recompute, and only then decide about rad. Don't drop both at once.

In [ ]:
df1 = df1.drop('tax',axis=1)

df1 = add_constant(df1)
# df1.columns
ouput_df = df['medv']
vifs = []
for i in range(df1.shape[1]):
	vif = variance_inflation_factor(df1.values,i)
	vifs.append(vif)

vif_df = pd.DataFrame({
	'feature' : df1.columns,
	'vifs' : vifs
})

vif_df

In [21]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(df1.drop('const',axis=1))

np.linalg.cond(X_scaled)

np.float64(5.874651640199523)

***Dropping a correlated feature loses only its unique information, since the shared part is still carried by the feature I keep, and I verify this by comparing cross-validated R² with and without it. If the feature has real unique signal, I'd keep it and use Ridge or Elastic Net, or switch to a tree-based model, which tolerates collinearity. I'd use PCA or PLS when there are many correlated features and interpretability doesn't matter. The right choice depends on whether the goal is prediction or explaining coefficients.***